In [ ]:
!pip3 install wikipedia-api -q
!pip install lancedb -q
!pip install tantivy -q

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

In [ ]:
from wikipediaapi import Wikipedia
wiki = Wikipedia('RAGBot/0.0', 'en')
doc = wiki.page('Rock_Lee').text
paragraphs = doc.split('\n\n')

In [ ]:
docs_embed= model.encode(paragraphs,normalize_embeddings=True)

In [ ]:
# Embedt h equery
query = "How did Rock Lee open his chakra gates?"
query_embed= model.encode(query,normalize_embeddings=True)

In [ ]:
import numpy as np
import torch

similarities= np.dot(docs_embed,query_embed.T)
# Convert the NumPy array to a PyTorch tensor
similarities_tensor = torch.tensor(similarities)
# Now apply topk to the tensor
top_3_idx= similarities_tensor.topk(3).indices.tolist()
most_similar_documents= [paragraphs[idx] for idx in top_3_idx]

In [ ]:
most_similar_documents

['Appearances\nIn Naruto\nRock Lee is a ninja from Konohagakure part of Team Guy, a four-man cell of ninja led by Might Guy. Inspired by Lee\'s determination to become stronger despite his inability to perform basic ninja techniques, Guy takes a personal interest in him, deciding to help him achieve his dream of becoming a powerful ninja by using only taijutsu that is primary hand-to-hand combat. This relationship with Guy causes Lee to acquire many of Guy\'s traits. Lee believes he can surpass the natural talents of others through hard work and passion; throughout the series, he attempts to surpass Neji Hyuga, who is labeled a "genius". Lee first appears in the series as a participant in the Chunin Exams, twice a year exams for ninja who wish to increase their rank. During the Chunin Exams, Lee battles Gaara, a ninja from the village of Sunagakure. In the fight, Lee opens the five of the eight chakra gates, limits on the body\'s ability to use chakra, using a forbidden technique known

In [ ]:
from wikipediaapi import Wikipedia
wiki = Wikipedia('RAGBot/0.0', 'en')
docs = [{'text': x, 'category': "anime"} for x in wiki.page("Rock_Lee").text.split('\n\n')]
docs += [{'text': x, 'category': "sports"} for x in wiki.page("Rock_Lee_(basketball)").text.split('\n\n')]

In [ ]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from lancedb.rerankers import CrossEncoderReranker

In [ ]:
model = get_registry().get("sentence-transformers").create(name="paraphrase-MiniLM-L6-v2")

In [ ]:
class Document(LanceModel):
  text: str = model.SourceField()
  vector: Vector(384) = model.VectorField()
  category: str
db = lancedb.connect('.my_db')
tbl = db.create_table("my_docv1", schema=Document)

In [ ]:
tbl.add(docs)
tbl.create_fts_index("text")

In [ ]:
reranker = CrossEncoderReranker()
query = "How did Rock Lee open his chakra gates?"

# Apply filter within the search query
results = (tbl.search(query, query_type='hybrid').where("category='anime'").limit(3).rerank(reranker=reranker))

In [ ]:
all_data = results.to_list()[:3]
[ele['text'] for ele in all_data[:3]] # get top3 results

['Appearances\nIn Naruto\nRock Lee is a ninja from Konohagakure part of Team Guy, a four-man cell of ninja led by Might Guy. Inspired by Lee\'s determination to become stronger despite his inability to perform basic ninja techniques, Guy takes a personal interest in him, deciding to help him achieve his dream of becoming a powerful ninja by using only taijutsu that is primary hand-to-hand combat. This relationship with Guy causes Lee to acquire many of Guy\'s traits. Lee believes he can surpass the natural talents of others through hard work and passion; throughout the series, he attempts to surpass Neji Hyuga, who is labeled a "genius". Lee first appears in the series as a participant in the Chunin Exams, twice a year exams for ninja who wish to increase their rank. During the Chunin Exams, Lee battles Gaara, a ninja from the village of Sunagakure. In the fight, Lee opens the five of the eight chakra gates, limits on the body\'s ability to use chakra, using a forbidden technique known